In [2]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [3]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [4]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 15:45:54, wtch_dt_end:2026-07-21 15:45:54


In [5]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [6]:
@file:DependsOn("org.json:json:20250107")

In [7]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [8]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Number,2142,1141,0,0.260000,49,4.981835,2.111342,0.250000,3.779000,5.450000,6.370000,14.858000
rtmWqChpla,Comparable<*>,2142,1325,0,,178,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2142,1,0,,2142,null,null,,,,,
rtmWqWtchStaCd,String,2142,14,0,SEA6001,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2142,2142,0,1,1,1071.500000,618.486459,1,535.916667,1071.500000,1607.083333,2142
rtmWqTu,Int,2142,151,0,5,190,24.573763,31.895511,0,5.000000,11.000000,32.000000,232
ph,Double,2142,127,0,7.630000,53,7.668772,0.264620,7.020000,7.490000,7.640000,7.910000,8.690000
rtmWqSlnty,Number,2142,1987,0,32.737999,4,21.822900,10.321351,0.284000,13.992000,26.639999,29.568001,34.032001
rtmWqCndctv,Float,2142,2065,0,45.320000,3,34.075322,15.468107,0.586000,23.501833,39.805500,45.179999,54.451000
rtmWqWtchDtlDt,String,2142,190,0,2026-07-20 16:25:00.0,14,null,null,2026-07-20 15:55:00.0,2026-07-20 21:45:00.0,2026-07-21 03:45:00.0,2026-07-21 09:40:00.0,2026-07-21 15:30:00.0


In [9]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  colsOf<Number>() }.with { it.toString().trim().toDouble()}


df.describe()

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2142,1137,0,0.260000,49,4.981835,2.111342,0.250000,3.778833,5.452000,6.370000,14.858000
rtmWqChpla,Double,2142,1325,0,0.000000,178,5.147720,5.216948,0.000000,1.307500,2.963000,7.910000,32.040000
rtmWqWtchStaCd,String,2142,14,0,SEA6001,178,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2142,2142,0,1.000000,1,1071.500000,618.486459,1.000000,535.916667,1071.500000,1607.083333,2142.000000
rtmWqTu,Double,2142,151,0,5.000000,190,24.573763,31.895511,0.000000,5.000000,11.000000,32.000000,232.000000
ph,Double,2142,127,0,7.630000,53,7.668772,0.264620,7.020000,7.490000,7.640000,7.910000,8.690000
rtmWqSlnty,Double,2142,1987,0,32.737999,4,21.822900,10.321351,0.284000,13.990666,26.641999,29.569584,34.032001
rtmWqCndctv,Double,2142,2065,0,45.320000,3,34.075322,15.468107,0.586000,23.501833,39.805500,45.180000,54.451000
rtmWqWtchDtlDt,LocalDateTime,2142,190,0,2026-07-20T16:25,14,null,null,2026-07-20T15:55,2026-07-20T21:45,2026-07-21T03:45,2026-07-21T09:40,2026-07-21T15:30
rtmWtchWtem,Double,2142,757,0,26.990000,14,26.391340,2.335509,19.740000,24.990000,26.459999,28.180834,30.889999


In [10]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")

In [11]:
val renamedDf = df.rename(
    "rtmWqDoxn" to "용존산소",
    "rtmWqChpla" to "클로로필",
    "rtmWqWtchStaCd" to "관측정점코드",
    "num" to "순번",
    "rtmWqTu" to "탁도",
    "ph" to "수소이온농도",
    "rtmWqSlnty" to "염분",
    "rtmWqCndctv" to "전기전도도",
    "rtmWqWtchDtlDt" to "일시",
    "rtmWtchWtem" to "수온"
)
renamedDf

용존산소,클로로필,관측정점코드,순번,탁도,수소이온농도,염분,전기전도도,일시,수온
6.350000,3.420000,NEP2002,1.000000,7.000000,7.980000,27.764000,42.621000,2026-07-20T15:55,24.379999
7.840000,1.980000,NEP1002,2.000000,26.000000,8.100000,1.268000,2.462000,2026-07-20T15:55,29.290001
6.520000,14.110000,SEA7002,3.000000,21.000000,7.540000,26.733000,40.186000,2026-07-20T15:55,23.150000
5.170000,0.560000,SEA1301,4.000000,6.000000,7.710000,30.763000,45.993000,2026-07-20T15:55,27.190001
3.257000,2.478000,SEA2007,5.000000,45.000000,7.880000,32.365002,52.207000,2026-07-20T15:55,24.340000
4.990000,8.030000,SEA5003,6.000000,12.000000,7.570000,32.680000,49.873000,2026-07-20T15:55,24.950001
1.030000,1.310000,SEA5002,7.000000,49.000000,7.490000,27.503000,42.927000,2026-07-20T15:55,29.250000
5.077000,1.426000,NEP3001,8.000000,21.000000,7.730000,17.066000,27.777000,2026-07-20T15:55,27.540001
7.287000,10.836000,SEA2005,9.000000,2.000000,7.920000,28.660000,44.392000,2026-07-20T15:55,30.379999
6.100000,10.680000,SEA6001,10.000000,44.000000,7.880000,30.149000,46.488000,2026-07-20T15:55,26.059999


In [12]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T15:55,6.350000,3.420000,NEP2002,7.000000,7.980000,27.764000,42.621000,24.379999
2.000000,2026-07-20T15:55,7.840000,1.980000,NEP1002,26.000000,8.100000,1.268000,2.462000,29.290001
3.000000,2026-07-20T15:55,6.520000,14.110000,SEA7002,21.000000,7.540000,26.733000,40.186000,23.150000
4.000000,2026-07-20T15:55,5.170000,0.560000,SEA1301,6.000000,7.710000,30.763000,45.993000,27.190001
5.000000,2026-07-20T15:55,3.257000,2.478000,SEA2007,45.000000,7.880000,32.365002,52.207000,24.340000
6.000000,2026-07-20T15:55,4.990000,8.030000,SEA5003,12.000000,7.570000,32.680000,49.873000,24.950001
7.000000,2026-07-20T15:55,1.030000,1.310000,SEA5002,49.000000,7.490000,27.503000,42.927000,29.250000
8.000000,2026-07-20T15:55,5.077000,1.426000,NEP3001,21.000000,7.730000,17.066000,27.777000,27.540001
9.000000,2026-07-20T15:55,7.287000,10.836000,SEA2005,2.000000,7.920000,28.660000,44.392000,30.379999
10.000000,2026-07-20T15:55,6.100000,10.680000,SEA6001,44.000000,7.880000,30.149000,46.488000,26.059999


In [16]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="GBH583" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("GBH583");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845629E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845632E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845638E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845641E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.7845647E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.784565E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845656E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845659E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845665E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845668E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845674E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845677E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845683E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845686E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845692E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845695E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845701E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7